## Export predictions for further validation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

class FinBERTRegression(nn.Module):
    def __init__(self, model_name="yiyanghkust/finbert-tone"):
        super(FinBERTRegression, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.regressor = nn.Linear(3, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        regression_score = self.regressor(outputs.logits)
        normalized_score = torch.tanh(regression_score)
        return normalized_score

class AdaptiveWeightedHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(AdaptiveWeightedHuberLoss, self).__init__()
        self.delta = delta

    def forward(self, predictions, targets, weights=None):
        error = predictions - targets
        abs_error = torch.abs(error)
        quadratic = torch.minimum(abs_error, torch.tensor(self.delta))
        linear = abs_error - quadratic
        loss = 0.5 * quadratic ** 2 + self.delta * linear
        if weights is not None:
            loss = loss * weights
        return loss.mean()



model = FinBERTRegression()
model.load_state_dict(torch.load(path+'fine-tuned-finbert.pth'))#, map_location=torch.device('cpu'))) #
model.eval()  # set to eval mode if you're not training

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
import numpy as np
import pandas as pd

news_df = pd.read_csv(path+"ok2.csv")
news_df

,Unnamed: 0,date,date_unixtime,text,from
0,0,2021-08-23T20:55:03,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru
1,1,2021-08-26T03:07:52,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru
2,2,2021-08-26T22:14:54,1630005294,Record breaking sale for Bored Ape,Watcher Guru
3,3,2021-09-02T20:58:54,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru
4,4,2021-09-10T03:05:01,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru
...,...,...,...,...,...
23360,25713,2025-11-07T02:38:24,1762475904,Jack Dorsey’s Block sees nearly $2 billion in ...,unfolded.
23361,25714,2025-11-07T07:31:30,1762493490,Japan's regulator backs joint stablecoin proje...,unfolded.
23362,25715,2025-11-07T14:30:36,1762518636,Tether moves nearly $100 million in Bitcoin to...,unfolded.
23363,25716,2025-11-07T14:35:12,1762518912,Kazakhstan to launch $1 billion crypto reserve...,unfolded.


In [ ]:
import torch
import pandas as pd
from tqdm.notebook import tqdm # Import tqdm


predicted_sentiments = []

# Wrap the loop with tqdm
for i, row in tqdm(news_df.iterrows(), total=len(news_df), desc="Predicting Sentiment"):
    timestamp = row['date_unixtime']
    text = row['text']
    source = row['from']
    #score = row['volume_surge_price'] # Although not used for prediction, keep for context if needed later

    # Inference
    inputs_inf = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    with torch.no_grad():
        sentiment = model(
            input_ids=inputs_inf["input_ids"],
            attention_mask=inputs_inf["attention_mask"]
        ).item()

    predicted_sentiments.append({
        'timestamp': timestamp,
        #'title': text,
        #'source': source,
        'predicted_sentiment': sentiment
    })

# Create DataFrame
predicted_df = pd.DataFrame(predicted_sentiments)

# Export to CSV
output_csv_path = path + "predicted_sentiments2.csv"
predicted_df.to_csv(output_csv_path, index=False)

print(f"Predicted sentiments exported to: {output_csv_path}")

Predicting Sentiment:   0%|          | 0/23365 [00:00<?, ?it/s]

Predicted sentiments exported to: /content/drive/MyDrive/Colab Notebooks/data/predicted_sentiments2.csv
